# 만족 여가활동 순위모형 EDA

- 01번 전처리 산출물을 불러와 순위 라벨과 입력 변수 구조를 점검함.
- 모델링용 결측 대체, 인코딩, train/valid/test 분리는 수행하지 않음.
- EDA 결과를 바탕으로 모델링용 전처리 판단사항을 정리함.

## 분석 환경 및 데이터 경로 설정

- `notebooks/preference` 기준 분석 경로를 설정함.
- 01번 산출물인 만족 순위 학습용 base 테이블과 중분류 매핑표 경로를 설정함.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", "{:,.4f}".format)

BASE_PATH = Path.cwd().resolve()

PREFERENCE_PATH = None
for path in [BASE_PATH, *BASE_PATH.parents]:
    if (path / "notebooks" / "preference" / "data").exists():
        PREFERENCE_PATH = path / "notebooks" / "preference"
        break

if PREFERENCE_PATH is None:
    raise FileNotFoundError("preference ?? ??? ?? ?????.")
DATA_PATH = PREFERENCE_PATH / "data"
PROCESSED_PATH = DATA_PATH / "processed" / "satisfaction"

RANK_BASE_PATH = PROCESSED_PATH / "ml_preference_satisfaction_rank_base.csv"
MAPPING_PATH = PROCESSED_PATH / "ml_activity_category_mapping.csv"

print("preference 경로:", PREFERENCE_PATH)
print("rank base 경로:", RANK_BASE_PATH)
print("mapping 경로:", MAPPING_PATH)

## 데이터 불러오기 및 기본 구조 확인

- 만족 순위 학습용 base 테이블을 불러옴.
- 여가활동 코드-문화누리 중분류 매핑표를 불러옴.
- 테이블 구조, 응답자 ID 중복, 중분류 후보군을 확인함.

In [ ]:
rank_base = pd.read_csv(RANK_BASE_PATH, encoding="utf-8-sig", low_memory=False)
activity_mapping = pd.read_csv(MAPPING_PATH, encoding="utf-8-sig")

rank_cols = [
    "만족_유효중분류_1순위",
    "만족_유효중분류_2순위",
    "만족_유효중분류_3순위",
]

valid_categories = (
    activity_mapping
    .loc[activity_mapping["학습타깃사용여부"], "중분류"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print("rank_base 구조:", rank_base.shape)
print("activity_mapping 구조:", activity_mapping.shape)
print("응답자_ID 중복:", rank_base["응답자_ID"].duplicated().sum())
print("중분류 후보군:", valid_categories)

display(rank_base.head())
display(activity_mapping.head(20))

## 코드 라벨 부여

- 성별, 연령, 시도, 지역규모 코드에 해석용 라벨을 부여함.
- 연령 코드는 설계서 기준 `70세 이상`까지 제공됨을 확인함.

In [ ]:
sex_map = {
    1: "남성",
    2: "여성",
}

age_map = {
    1: "15-19세",
    2: "20대",
    3: "30대",
    4: "40대",
    5: "50대",
    6: "60대",
    7: "70세 이상",
}

sido_map = {
    1: "서울",
    2: "부산",
    3: "대구",
    4: "인천",
    5: "광주",
    6: "대전",
    7: "울산",
    8: "세종",
    9: "경기",
    10: "강원",
    11: "충북",
    12: "충남",
    13: "전북",
    14: "전남",
    15: "경북",
    16: "경남",
    17: "제주",
}

region_size_map = {
    1: "대도시",
    2: "중소도시",
    3: "읍면지역",
}

eda_df = rank_base.copy()

eda_df["성별_라벨"] = eda_df["성별"].map(sex_map)
eda_df["연령대"] = eda_df["연령"].map(age_map)
eda_df["시도"] = eda_df["17개 시도"].map(sido_map)
eda_df["지역규모_라벨"] = eda_df["지역규모"].map(region_size_map)

label_check = pd.DataFrame({
    "칼럼명": ["성별_라벨", "연령대", "시도", "지역규모_라벨"],
    "라벨결측": [
        eda_df["성별_라벨"].isna().sum(),
        eda_df["연령대"].isna().sum(),
        eda_df["시도"].isna().sum(),
        eda_df["지역규모_라벨"].isna().sum(),
    ]
})

display(label_check)
print("연령대 분포")
print(eda_df["연령대"].value_counts(dropna=False).sort_index())

## 순위 라벨 품질 점검

- 유효 순위 수 분포를 확인함.
- 제외 분류 제거 수와 중복 중분류 제거 수를 확인함.
- 순위별 결측률을 확인함.

In [ ]:
rank_quality = pd.DataFrame({
    "점검항목": [
        "전체 응답자 수",
        "1순위 결측",
        "2순위 결측",
        "3순위 결측",
    ],
    "값": [
        len(eda_df),
        eda_df["만족_유효중분류_1순위"].isna().sum(),
        eda_df["만족_유효중분류_2순위"].isna().sum(),
        eda_df["만족_유효중분류_3순위"].isna().sum(),
    ]
})

print("유효 순위 수 분포")
print(eda_df["만족_유효순위수"].value_counts().sort_index())

print("\n제외 분류 제거 수 분포")
print(eda_df["제외분류_제거수"].value_counts().sort_index())

print("\n중복 중분류 제거 수 분포")
print(eda_df["중복중분류_제거수"].value_counts().sort_index())

display(rank_quality)

## 순위별 중분류 분포 확인

- 1순위, 2순위, 3순위별 중분류 빈도와 비율을 확인함.
- 중분류별 Top-3 포함률을 확인함.

In [ ]:
rank_distribution_list = []

for rank, col in enumerate(rank_cols, start=1):
    temp = (
        eda_df[col]
        .value_counts(dropna=False)
        .reset_index()
    )
    temp.columns = ["중분류", "빈도"]
    temp["순위"] = rank
    temp["비율"] = temp["빈도"] / len(eda_df)
    rank_distribution_list.append(temp)

rank_distribution = pd.concat(rank_distribution_list, ignore_index=True)
rank_distribution = rank_distribution[["순위", "중분류", "빈도", "비율"]]

print("순위별 중분류 분포")
display(rank_distribution)

top3_rows = []

for category in valid_categories:
    include_count = eda_df[rank_cols].eq(category).any(axis=1).sum()
    top3_rows.append({
        "중분류": category,
        "Top3_포함응답자수": include_count,
        "Top3_포함률": include_count / len(eda_df),
    })

top3_include = (
    pd.DataFrame(top3_rows)
    .sort_values("Top3_포함률", ascending=False)
)

print("\n중분류별 Top-3 포함률")
display(top3_include)

## 최종가중치 적용 전후 분포 확인

- 1순위 중분류 분포를 비가중 기준으로 계산함.
- 1순위 중분류 분포를 최종가중치 적용 기준으로 계산함.
- 가중치 적용 전후 비율 차이를 확인함.

In [ ]:
rank1_col = "만족_유효중분류_1순위"

unweighted_rank1 = (
    eda_df[rank1_col]
    .value_counts(normalize=True)
    .rename("비가중_비율")
    .reset_index()
    .rename(columns={rank1_col: "중분류"})
)

weighted_rank1 = (
    eda_df
    .groupby(rank1_col, as_index=False)["최종가중치"]
    .sum()
    .rename(columns={rank1_col: "중분류", "최종가중치": "가중합"})
)
weighted_rank1["가중_비율"] = weighted_rank1["가중합"] / weighted_rank1["가중합"].sum()

weight_compare = (
    unweighted_rank1
    .merge(weighted_rank1[["중분류", "가중_비율"]], on="중분류", how="outer")
    .fillna(0)
)
weight_compare["가중-비가중"] = weight_compare["가중_비율"] - weight_compare["비가중_비율"]
weight_compare = weight_compare.sort_values("가중-비가중", ascending=False)

display(weight_compare)

## 1순위-2순위 전이 패턴 확인

- 1순위 중분류별 2순위 중분류 빈도를 확인함.
- 1순위 중분류별 2순위 비율을 확인함.

In [ ]:
transition_count = pd.crosstab(
    eda_df["만족_유효중분류_1순위"],
    eda_df["만족_유효중분류_2순위"],
    dropna=False
)

transition_ratio = pd.crosstab(
    eda_df["만족_유효중분류_1순위"],
    eda_df["만족_유효중분류_2순위"],
    normalize="index",
    dropna=False
)

print("1순위-2순위 전이 빈도")
display(transition_count)

print("\n1순위-2순위 전이 비율")
display(transition_ratio)

## Pairwise Preference Matrix 생성

- 한 응답자의 앞 순위 중분류가 뒤 순위 중분류보다 선호된 것으로 집계함.
- 행 중분류가 열 중분류보다 앞선 횟수를 계산함.
- 중분류 쌍별 비교 정보가 없는 조합을 확인함.

In [ ]:
pairwise_matrix = pd.DataFrame(
    0,
    index=valid_categories,
    columns=valid_categories,
    dtype=int
)

for idx, row in eda_df[rank_cols].iterrows():
    ordered_categories = [category for category in row.tolist() if pd.notna(category)]
    
    for i in range(len(ordered_categories)):
        for j in range(i + 1, len(ordered_categories)):
            winner = ordered_categories[i]
            loser = ordered_categories[j]
            pairwise_matrix.loc[winner, loser] += 1

pairwise_total = pairwise_matrix + pairwise_matrix.T

comparison_summary_rows = []

for category in valid_categories:
    total_compare = pairwise_total.loc[category].sum()
    total_win = pairwise_matrix.loc[category].sum()
    
    comparison_summary_rows.append({
        "중분류": category,
        "앞선횟수": total_win,
        "전체비교횟수": total_compare,
        "pairwise_승률": total_win / total_compare if total_compare > 0 else np.nan,
    })

comparison_summary = (
    pd.DataFrame(comparison_summary_rows)
    .sort_values("pairwise_승률", ascending=False)
)

zero_pair_count = 0

for i, category_a in enumerate(valid_categories):
    for category_b in valid_categories[i + 1:]:
        if pairwise_total.loc[category_a, category_b] == 0:
            zero_pair_count += 1

print("Pairwise Preference Matrix")
display(pairwise_matrix)

print("\n중분류별 pairwise 비교 요약")
display(comparison_summary)

print("\n비교 정보가 없는 중분류 쌍 수:", zero_pair_count)

## 입력 변수 결측치 확인

- baseline 직접 입력 변수의 결측치를 확인함.
- 후보 확장 변수의 결측치를 확인함.
- 결측 대체는 수행하지 않음.

In [ ]:
direct_feature_cols = [
    "성별",
    "연령",
    "17개 시도",
    "지역규모",
    "조사년도",
]

candidate_feature_cols = [
    "학력",
    "가구소득",
    "장애여부",
    "여가활동을 위한 월평균 지출액",
    "적절하다고 생각하는 월평균 여가비용",
    "평일 하루 평균 여가시간",
    "휴일 하루 평균 여가시간",
    "생활권 내 공공문화여가시설 이용 충분도",
    "생활권 내 공공문화여가시설 이용 여부",
    "생활권 내 공공문화여가시설 만족도",
    "여가활동 제약요인_시간부족",
    "여가활동 제약요인_경제적 지출 부담",
    "여가활동 제약요인_여가활동 경험 부족",
    "여가활동 제약요인_여가 정보 부족",
    "여가활동 제약요인_질병 및 장애",
    "여가활동 제약요인_여가 동반자 없음",
    "여가활동 제약요인_여가시설 접근성 부족",
    "여가활동 제약요인_여가프로그램 부족",
]

def missing_table(data, columns):
    result = (
        data[columns]
        .isna()
        .sum()
        .reset_index(name="결측치")
        .rename(columns={"index": "칼럼명"})
    )
    result["결측률"] = result["결측치"] / len(data)
    return result.sort_values("결측률", ascending=False)

direct_missing = missing_table(eda_df, direct_feature_cols)
candidate_missing = missing_table(eda_df, candidate_feature_cols)

print("baseline 직접 입력 변수 결측치")
display(direct_missing)

print("\n후보 확장 변수 결측치")
display(candidate_missing)

## 입력 변수 분포 확인

- baseline 직접 입력 변수의 value 분포를 확인함.
- 조사년도별 1순위 중분류 분포를 확인함.
- 성별, 연령대, 지역규모별 1순위 중분류 분포를 확인함.

In [ ]:
print("성별 분포")
print(eda_df["성별_라벨"].value_counts(dropna=False))

print("\n연령대 분포")
print(eda_df["연령대"].value_counts(dropna=False).sort_index())

print("\n시도 분포")
print(eda_df["시도"].value_counts(dropna=False))

print("\n지역규모 분포")
print(eda_df["지역규모_라벨"].value_counts(dropna=False))

print("\n조사년도별 1순위 중분류 분포")
display(pd.crosstab(eda_df["조사년도"], eda_df[rank1_col], normalize="index"))

print("\n성별별 1순위 중분류 분포")
display(pd.crosstab(eda_df["성별_라벨"], eda_df[rank1_col], normalize="index"))

print("\n연령대별 1순위 중분류 분포")
display(pd.crosstab(eda_df["연령대"], eda_df[rank1_col], normalize="index"))

print("\n지역규모별 1순위 중분류 분포")
display(pd.crosstab(eda_df["지역규모_라벨"], eda_df[rank1_col], normalize="index"))

## Top-3 포함률 그룹별 비교

- 성별, 연령대, 지역규모별 Top-3 포함률을 계산함.
- 각 그룹에서 중분류가 유효 1~3순위 안에 포함된 비율을 확인함.

In [ ]:
def top3_rate_by_group(data, group_col):
    result_list = []
    
    for group_value, group_data in data.groupby(group_col, dropna=False):
        group_size = len(group_data)
        
        for category in valid_categories:
            include_rate = group_data[rank_cols].eq(category).any(axis=1).sum() / group_size
            result_list.append({
                group_col: group_value,
                "중분류": category,
                "Top3_포함률": include_rate,
                "응답자수": group_size,
            })
    
    return pd.DataFrame(result_list)

gender_top3 = top3_rate_by_group(eda_df, "성별_라벨")
age_top3 = top3_rate_by_group(eda_df, "연령대")
region_top3 = top3_rate_by_group(eda_df, "지역규모_라벨")

print("성별별 Top-3 포함률")
display(gender_top3.pivot(index="성별_라벨", columns="중분류", values="Top3_포함률"))

print("\n연령대별 Top-3 포함률")
display(age_top3.pivot(index="연령대", columns="중분류", values="Top3_포함률"))

print("\n지역규모별 Top-3 포함률")
display(region_top3.pivot(index="지역규모_라벨", columns="중분류", values="Top3_포함률"))

## EDA 결과 정리

- 결측 대체 전 직접 입력 변수 결측 결과를 확인함.
- 순위 라벨 분포와 pairwise 비교 정보를 확인함.
- 모델링용 전처리는 다음 단계에서 수행함.

## 모델링용 입력 변수 확정

- baseline 직접 대응 입력 변수만 사용함.
- 성별, 연령대, 시도, 지역규모, 조사년도를 입력 변수로 설정함.
- 직접 입력 변수 결측치가 없어 결측 대체를 수행하지 않음.
- 최종가중치를 평균 1로 표준화하여 학습 가중치로 사용함.

In [ ]:
model_df = eda_df.copy()

model_feature_cols = [
    "성별_라벨",
    "연령대",
    "시도",
    "지역규모_라벨",
    "조사년도",
]

model_df["조사년도_라벨"] = model_df["조사년도"].astype(str)
model_feature_cols = [
    "성별_라벨",
    "연령대",
    "시도",
    "지역규모_라벨",
    "조사년도_라벨",
]

model_missing = (
    model_df[model_feature_cols + rank_cols + ["최종가중치"]]
    .isna()
    .sum()
    .reset_index(name="결측치")
    .rename(columns={"index": "칼럼명"})
)

model_missing["결측률"] = model_missing["결측치"] / len(model_df)

print("모델링 대상 구조:", model_df.shape)
print("입력 변수:", model_feature_cols)
display(model_missing)

## Train / Valid / Test 분리

- 응답자 단위로 train, valid, test를 분리함.
- 조사년도와 1순위 중분류 조합을 기준으로 층화 분리함.
- train 70%, valid 15%, test 15% 비율로 분리함.

In [ ]:
from sklearn.model_selection import train_test_split

split_key = (
    model_df["조사년도_라벨"]
    + "_"
    + model_df["만족_유효중분류_1순위"]
)

train_valid_idx, test_idx = train_test_split(
    model_df.index,
    test_size=0.15,
    random_state=42,
    stratify=split_key,
)

train_valid_df = model_df.loc[train_valid_idx].copy()
train_valid_key = (
    train_valid_df["조사년도_라벨"]
    + "_"
    + train_valid_df["만족_유효중분류_1순위"]
)

train_idx, valid_idx = train_test_split(
    train_valid_df.index,
    test_size=0.15 / 0.85,
    random_state=42,
    stratify=train_valid_key,
)

train_df = model_df.loc[train_idx].copy()
valid_df = model_df.loc[valid_idx].copy()
test_df = model_df.loc[test_idx].copy()

split_summary = pd.DataFrame({
    "데이터": ["train", "valid", "test"],
    "응답자수": [len(train_df), len(valid_df), len(test_df)],
    "비율": [len(train_df) / len(model_df), len(valid_df) / len(model_df), len(test_df) / len(model_df)],
})

display(split_summary)

print("조사년도 분포")
display(pd.crosstab(
    model_df.loc[train_df.index.union(valid_df.index).union(test_df.index), "조사년도"],
    pd.Series(
        np.select(
            [
                model_df.index.isin(train_df.index),
                model_df.index.isin(valid_df.index),
                model_df.index.isin(test_df.index),
            ],
            ["train", "valid", "test"],
            default="other",
        ),
        index=model_df.index,
        name="split"
    ),
    normalize="columns"
))

print("1순위 중분류 분포")
split_label_df = pd.concat([
    train_df.assign(split="train"),
    valid_df.assign(split="valid"),
    test_df.assign(split="test"),
])

display(pd.crosstab(
    split_label_df["만족_유효중분류_1순위"],
    split_label_df["split"],
    normalize="columns"
))

## Rank-Ordered Logit 학습 배열 생성

- 범주형 입력 변수를 더미 변수로 변환함.
- 만족 유효 1~3순위를 중분류 인덱스로 변환함.
- 학습 가중치를 평균 1 기준으로 표준화함.

In [ ]:
from scipy.optimize import minimize
from scipy.special import logsumexp

all_feature_df = pd.get_dummies(
    model_df[model_feature_cols],
    drop_first=True,
    dtype=float
)

all_feature_df.insert(0, "상수", 1.0)
feature_columns = all_feature_df.columns.tolist()

category_to_idx = {
    category: idx
    for idx, category in enumerate(valid_categories)
}

idx_to_category = {
    idx: category
    for category, idx in category_to_idx.items()
}

reference_category = "영상"
reference_idx = category_to_idx[reference_category]
nonref_idx = [
    idx for idx in range(len(valid_categories))
    if idx != reference_idx
]

def make_model_arrays(data):
    X = all_feature_df.loc[data.index].to_numpy(dtype=float)
    y = np.full((len(data), 3), -1, dtype=int)
    
    for rank, col in enumerate(rank_cols):
        y[:, rank] = (
            data[col]
            .map(category_to_idx)
            .fillna(-1)
            .astype(int)
            .to_numpy()
        )
    
    sample_weight = data["최종가중치"].to_numpy(dtype=float)
    sample_weight = sample_weight / np.nanmean(sample_weight)
    
    return X, y, sample_weight

X_train, y_train, weight_train = make_model_arrays(train_df)
X_valid, y_valid, weight_valid = make_model_arrays(valid_df)
X_test, y_test, weight_test = make_model_arrays(test_df)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test:", X_test.shape)
print("입력 변수 더미 수:", len(feature_columns))
print("기준 중분류:", reference_category)

## Rank-Ordered Logit 모델 함수 정의

- 순위 단계별 후보 중분류 집합을 구성함.
- 선택된 중분류의 로그우도를 계산함.
- L-BFGS-B 최적화에 사용할 목적함수와 gradient를 정의함.

In [ ]:
K = len(valid_categories)
D = X_train.shape[1]
l2_alpha = 1e-4

choice_weight_sum = sum(
    weight_train[y_train[:, rank] >= 0].sum()
    for rank in range(3)
)

def unpack_params(params):
    W_nonref = params.reshape(D, K - 1)
    W = np.zeros((D, K), dtype=float)
    W[:, nonref_idx] = W_nonref
    return W

def loss_grad(params):
    W = unpack_params(params)
    scores = X_train @ W
    grad_scores = np.zeros_like(scores)
    loss = 0.0
    
    for rank in range(3):
        valid_mask = y_train[:, rank] >= 0
        
        if valid_mask.sum() == 0:
            continue
        
        valid_idx = np.where(valid_mask)[0]
        chosen = y_train[valid_idx, rank]
        
        masked_scores = scores[valid_idx].copy()
        remain_mask = np.ones_like(masked_scores, dtype=bool)
        
        for prev_rank in range(rank):
            prev_chosen = y_train[valid_idx, prev_rank]
            remain_mask[np.arange(len(valid_idx)), prev_chosen] = False
        
        masked_scores[~remain_mask] = -np.inf
        log_den = logsumexp(masked_scores, axis=1)
        probs = np.exp(masked_scores - log_den[:, None])
        probs[~remain_mask] = 0
        
        row_grad = probs
        row_grad[np.arange(len(valid_idx)), chosen] -= 1
        
        row_weight = weight_train[valid_idx]
        loss += np.sum(row_weight * (log_den - scores[valid_idx, chosen]))
        grad_scores[valid_idx] += row_grad * row_weight[:, None]
    
    grad_full = X_train.T @ grad_scores
    grad_nonref = grad_full[:, nonref_idx]
    params_nonref = W[:, nonref_idx].ravel()
    
    loss = loss / choice_weight_sum + 0.5 * l2_alpha * np.sum(params_nonref ** 2)
    grad = (grad_nonref / choice_weight_sum).ravel() + l2_alpha * params_nonref
    
    return loss, grad

## Rank-Ordered Logit 모델 학습

- L-BFGS-B 알고리즘으로 모델 파라미터를 추정함.
- 약한 L2 정규화를 적용함.
- 최적화 수렴 여부와 반복 횟수를 확인함.

In [ ]:
init_params = np.zeros(D * (K - 1), dtype=float)

rol_result = minimize(
    fun=lambda params: loss_grad(params),
    x0=init_params,
    jac=True,
    method="L-BFGS-B",
    options={"maxiter": 500},
)

W_hat = unpack_params(rol_result.x)

print("수렴 여부:", rol_result.success)
print("수렴 메시지:", rol_result.message)
print("반복 횟수:", rol_result.nit)
print("최종 목적함수:", rol_result.fun)

## 성능 평가 함수 정의

- Top-1 Accuracy를 계산함.
- Top-3 Hit Rate를 계산함.
- MRR을 계산함.
- NDCG@3을 계산함.
- Rank-Ordered Logit Log Loss를 계산함.

In [ ]:
def predict_prob(X, W):
    scores = X @ W
    scores = scores - scores.max(axis=1, keepdims=True)
    exp_scores = np.exp(scores)
    return exp_scores / exp_scores.sum(axis=1, keepdims=True)

def rol_log_loss(X, y, W, sample_weight=None):
    if sample_weight is None:
        sample_weight = np.ones(X.shape[0])
    
    scores = X @ W
    total_loss = 0.0
    total_weight = 0.0
    
    for rank in range(3):
        valid_mask = y[:, rank] >= 0
        
        if valid_mask.sum() == 0:
            continue
        
        valid_idx = np.where(valid_mask)[0]
        chosen = y[valid_idx, rank]
        
        masked_scores = scores[valid_idx].copy()
        remain_mask = np.ones_like(masked_scores, dtype=bool)
        
        for prev_rank in range(rank):
            prev_chosen = y[valid_idx, prev_rank]
            remain_mask[np.arange(len(valid_idx)), prev_chosen] = False
        
        masked_scores[~remain_mask] = -np.inf
        log_den = logsumexp(masked_scores, axis=1)
        
        row_weight = sample_weight[valid_idx]
        total_loss += np.sum(row_weight * (log_den - scores[valid_idx, chosen]))
        total_weight += row_weight.sum()
    
    return total_loss / total_weight

def ndcg_at_3(prob, y):
    pred_order = np.argsort(-prob, axis=1)[:, :3]
    ndcg_list = []
    
    for i in range(len(y)):
        relevance = {}
        
        for rank in range(3):
            if y[i, rank] >= 0:
                relevance[y[i, rank]] = 3 - rank
        
        if len(relevance) == 0:
            continue
        
        dcg = 0.0
        
        for position, category_idx in enumerate(pred_order[i], start=1):
            rel = relevance.get(category_idx, 0)
            dcg += (2 ** rel - 1) / np.log2(position + 1)
        
        ideal_relevance = sorted(relevance.values(), reverse=True)[:3]
        idcg = sum(
            (2 ** rel - 1) / np.log2(position + 1)
            for position, rel in enumerate(ideal_relevance, start=1)
        )
        
        ndcg_list.append(dcg / idcg if idcg > 0 else np.nan)
    
    return np.nanmean(ndcg_list)

def evaluate_rank_model(data_name, X, y, W, sample_weight=None):
    prob = predict_prob(X, W)
    pred_order = np.argsort(-prob, axis=1)
    rank1 = y[:, 0]
    
    top1_accuracy = np.mean(pred_order[:, 0] == rank1)
    top3_hit_rate = np.mean([
        rank1[i] in pred_order[i, :3]
        for i in range(len(rank1))
    ])
    mrr = np.mean([
        1 / (np.where(pred_order[i] == rank1[i])[0][0] + 1)
        for i in range(len(rank1))
    ])
    
    return {
        "데이터": data_name,
        "Top1_Accuracy": top1_accuracy,
        "Top3_HitRate": top3_hit_rate,
        "MRR": mrr,
        "NDCG@3": ndcg_at_3(prob, y),
        "ROL_LogLoss": rol_log_loss(X, y, W, sample_weight),
    }

## 기준모델 및 학습모델 성능 평가

- train 1순위 중분류 분포 기반 기준모델을 생성함.
- Rank-Ordered Logit 학습모델 성능을 계산함.
- 기준모델과 학습모델의 train, valid, test 성능을 비교함.

In [ ]:
train_prior = (
    train_df["만족_유효중분류_1순위"]
    .value_counts(normalize=True)
)

prior_prob = np.array([
    train_prior.get(category, 0)
    for category in valid_categories
])

prior_prob = prior_prob / prior_prob.sum()
prior_scores = np.log(prior_prob + 1e-12)

W_prior = np.zeros_like(W_hat)
W_prior[0, :] = prior_scores - prior_scores[reference_idx]

performance_table = pd.DataFrame([
    evaluate_rank_model("train_model", X_train, y_train, W_hat),
    evaluate_rank_model("valid_model", X_valid, y_valid, W_hat),
    evaluate_rank_model("test_model", X_test, y_test, W_hat),
    evaluate_rank_model("train_prior", X_train, y_train, W_prior),
    evaluate_rank_model("valid_prior", X_valid, y_valid, W_prior),
    evaluate_rank_model("test_prior", X_test, y_test, W_prior),
])

display(performance_table)

## 다항 로지스틱 학습
- 1순위 중분류를 목표변수로 설정함.
- ROL과 동일한 입력변수, 데이터 분할, 최종가중치를 사용함.
- 1순위 예측 기준의 비교모형을 생성함.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

mnl_feature_df = all_feature_df.drop(columns="상수").copy()
mnl_feature_columns = mnl_feature_df.columns.tolist()

X_train_mnl = mnl_feature_df.loc[train_df.index].to_numpy(dtype=float)
X_valid_mnl = mnl_feature_df.loc[valid_df.index].to_numpy(dtype=float)
X_test_mnl = mnl_feature_df.loc[test_df.index].to_numpy(dtype=float)

y_train_mnl = train_df["만족_유효중분류_1순위"].to_numpy()
y_valid_mnl = valid_df["만족_유효중분류_1순위"].to_numpy()
y_test_mnl = test_df["만족_유효중분류_1순위"].to_numpy()

mnl_model = LogisticRegression(
    solver="lbfgs",
    max_iter=1000,
    C=1.0,
)

mnl_model.fit(
    X_train_mnl,
    y_train_mnl,
    sample_weight=weight_train,
)

print("수렴 반복 횟수:", mnl_model.n_iter_[0])
print("입력 더미 변수 수:", len(mnl_feature_columns))
print("예측 중분류 수:", len(mnl_model.classes_))
print("예측 중분류:", sorted(mnl_model.classes_))

## 다항 로지스틱 성능 평가
- ROL 평가와 동일하게 Top1, Top3, MRR, NDCG@3를 산출함.
- 다항 로지스틱의 LogLoss는 1순위 중분류 확률 기준으로 산출함.
- prior 기준모형과 비교 가능한 성능표를 생성함.

In [ ]:
def align_mnl_prob(model, X):
    raw_prob = model.predict_proba(X)
    prob = np.zeros((X.shape[0], len(valid_categories)))
    class_to_col = {
        category: idx
        for idx, category in enumerate(model.classes_)
    }
    
    for j, category in enumerate(valid_categories):
        if category in class_to_col:
            prob[:, j] = raw_prob[:, class_to_col[category]]
    
    row_sum = prob.sum(axis=1, keepdims=True)
    prob = np.divide(
        prob,
        row_sum,
        out=np.zeros_like(prob),
        where=row_sum > 0,
    )
    
    return prob


def evaluate_prob_rank_model(data_name, prob, y, y_label):
    pred_order = np.argsort(-prob, axis=1)
    rank1 = y[:, 0]
    
    top1_accuracy = np.mean(pred_order[:, 0] == rank1)
    top3_hit_rate = np.mean([
        rank1[i] in pred_order[i, :3]
        for i in range(len(rank1))
    ])
    mrr = np.mean([
        1 / (np.where(pred_order[i] == rank1[i])[0][0] + 1)
        for i in range(len(rank1))
    ])
    ndcg = ndcg_at_3(prob, y)
    rank1_logloss = log_loss(
        y_label,
        prob,
        labels=valid_categories,
    )
    
    return {
        "데이터": data_name,
        "Top1_Accuracy": top1_accuracy,
        "Top3_HitRate": top3_hit_rate,
        "MRR": mrr,
        "NDCG@3": ndcg,
        "Rank1_LogLoss": rank1_logloss,
    }

mnl_prob_train = align_mnl_prob(mnl_model, X_train_mnl)
mnl_prob_valid = align_mnl_prob(mnl_model, X_valid_mnl)
mnl_prob_test = align_mnl_prob(mnl_model, X_test_mnl)

mnl_prior_train = np.tile(prior_prob, (len(train_df), 1))
mnl_prior_valid = np.tile(prior_prob, (len(valid_df), 1))
mnl_prior_test = np.tile(prior_prob, (len(test_df), 1))

mnl_performance_table = pd.DataFrame([
    evaluate_prob_rank_model("train_model", mnl_prob_train, y_train, y_train_mnl),
    evaluate_prob_rank_model("valid_model", mnl_prob_valid, y_valid, y_valid_mnl),
    evaluate_prob_rank_model("test_model", mnl_prob_test, y_test, y_test_mnl),
    evaluate_prob_rank_model("train_prior", mnl_prior_train, y_train, y_train_mnl),
    evaluate_prob_rank_model("valid_prior", mnl_prior_valid, y_valid, y_valid_mnl),
    evaluate_prob_rank_model("test_prior", mnl_prior_test, y_test, y_test_mnl),
])

mnl_performance_table = mnl_performance_table.round(4)
display(mnl_performance_table)

rol_compare = performance_table.copy().rename(columns={"ROL_LogLoss": "LogLoss"})
rol_compare["모델"] = "ROL"
rol_compare["LogLoss_기준"] = "순위"

mnl_compare = mnl_performance_table.copy().rename(columns={"Rank1_LogLoss": "LogLoss"})
mnl_compare["모델"] = "다항로지스틱"
mnl_compare["LogLoss_기준"] = "1순위"

model_performance_compare = pd.concat([
    rol_compare,
    mnl_compare,
], ignore_index=True)

model_performance_compare = model_performance_compare[[
    "모델",
    "데이터",
    "Top1_Accuracy",
    "Top3_HitRate",
    "MRR",
    "NDCG@3",
    "LogLoss",
    "LogLoss_기준",
]]

model_performance_compare = model_performance_compare.round(4)
display(model_performance_compare)

## 모델별 계수 테이블 생성
- ROL의 중분류별 계수 테이블을 생성함.
- 다항 로지스틱의 중분류별 계수 테이블을 생성함.
- 계수 절대값 기준 상위 입력변수를 확인함.

In [ ]:
rol_coef_table = pd.DataFrame(
    W_hat,
    index=feature_columns,
    columns=valid_categories,
).reset_index().rename(columns={"index": "입력변수"})

rol_coef_long = rol_coef_table.melt(
    id_vars="입력변수",
    var_name="중분류",
    value_name="계수",
)

rol_coef_long["계수절대값"] = rol_coef_long["계수"].abs()
rol_coef_top = (
    rol_coef_long[rol_coef_long["입력변수"] != "상수"]
    .sort_values("계수절대값", ascending=False)
    .head(30)
)

print("ROL 계수 절대값 상위 30개")
display(rol_coef_top)

mnl_coef_table = pd.DataFrame(
    mnl_model.coef_,
    index=mnl_model.classes_,
    columns=mnl_feature_columns,
).reset_index().rename(columns={"index": "중분류"})

mnl_coef_long = mnl_coef_table.melt(
    id_vars="중분류",
    var_name="입력변수",
    value_name="계수",
)

mnl_coef_long["계수절대값"] = mnl_coef_long["계수"].abs()
mnl_coef_top = (
    mnl_coef_long
    .sort_values("계수절대값", ascending=False)
    .head(30)
)

print("다항 로지스틱 계수 절대값 상위 30개")
display(mnl_coef_top)

## 변수군 중요도 점검
- 검증셋에서 변수군 값을 무작위로 섞은 뒤 손실 증가량을 계산함.
- 손실 증가량이 클수록 해당 변수군이 예측 성능에 더 크게 기여함.
- ROL은 순위 LogLoss, 다항 로지스틱은 1순위 LogLoss를 기준으로 계산함.

In [ ]:
def make_feature_group_map(columns):
    return {
        "성별": [col for col in columns if col.startswith("성별_라벨_")],
        "연령대": [col for col in columns if col.startswith("연령대_")],
        "시도": [col for col in columns if col.startswith("시도_")],
        "지역규모": [col for col in columns if col.startswith("지역규모_라벨_")],
        "조사년도": [col for col in columns if col.startswith("조사년도_라벨_")],
    }


def permutation_importance_rol(repeat_count=3):
    rng = np.random.default_rng(42)
    base_loss = rol_log_loss(X_valid, y_valid, W_hat)
    group_map = make_feature_group_map(feature_columns)
    rows = []
    
    for group_name, group_cols in group_map.items():
        col_idx = [feature_columns.index(col) for col in group_cols]
        losses = []
        
        if len(col_idx) == 0:
            continue
        
        for _ in range(repeat_count):
            X_perm = X_valid.copy()
            perm_idx = rng.permutation(X_perm.shape[0])
            X_perm[:, col_idx] = X_perm[perm_idx][:, col_idx]
            losses.append(rol_log_loss(X_perm, y_valid, W_hat))
        
        rows.append({
            "모델": "ROL",
            "변수군": group_name,
            "기준손실": base_loss,
            "섞은후손실": np.mean(losses),
            "손실증가량": np.mean(losses) - base_loss,
        })
    
    return pd.DataFrame(rows)


def permutation_importance_mnl(repeat_count=3):
    rng = np.random.default_rng(42)
    base_loss = log_loss(y_valid_mnl, mnl_prob_valid, labels=valid_categories)
    group_map = make_feature_group_map(mnl_feature_columns)
    rows = []
    
    for group_name, group_cols in group_map.items():
        col_idx = [mnl_feature_columns.index(col) for col in group_cols]
        losses = []
        
        if len(col_idx) == 0:
            continue
        
        for _ in range(repeat_count):
            X_perm = X_valid_mnl.copy()
            perm_idx = rng.permutation(X_perm.shape[0])
            X_perm[:, col_idx] = X_perm[perm_idx][:, col_idx]
            perm_prob = align_mnl_prob(mnl_model, X_perm)
            losses.append(log_loss(y_valid_mnl, perm_prob, labels=valid_categories))
        
        rows.append({
            "모델": "다항로지스틱",
            "변수군": group_name,
            "기준손실": base_loss,
            "섞은후손실": np.mean(losses),
            "손실증가량": np.mean(losses) - base_loss,
        })
    
    return pd.DataFrame(rows)

importance_table = pd.concat([
    permutation_importance_rol(),
    permutation_importance_mnl(),
], ignore_index=True)

importance_table = importance_table.sort_values(
    ["모델", "손실증가량"],
    ascending=[True, False],
).round(6)

display(importance_table)

## 예측 분포 점검
- 검증셋의 실제 1순위 분포와 모델별 예측 1순위 분포를 비교함.
- 특정 중분류로 예측이 쏠리는지 확인함.

In [ ]:
def make_prediction_distribution(prob, data_name):
    pred_idx = np.argmax(prob, axis=1)
    pred_label = [idx_to_category[idx] for idx in pred_idx]
    
    return (
        pd.Series(pred_label)
        .value_counts(normalize=True)
        .rename_axis("중분류")
        .reset_index(name=data_name)
    )

actual_valid_distribution = (
    valid_df["만족_유효중분류_1순위"]
    .value_counts(normalize=True)
    .rename_axis("중분류")
    .reset_index(name="실제_1순위_비율")
)

rol_valid_distribution = make_prediction_distribution(
    predict_prob(X_valid, W_hat),
    "ROL_예측_1순위_비율",
)

mnl_valid_distribution = make_prediction_distribution(
    mnl_prob_valid,
    "다항로지스틱_예측_1순위_비율",
)

prediction_distribution_compare = (
    actual_valid_distribution
    .merge(rol_valid_distribution, on="중분류", how="outer")
    .merge(mnl_valid_distribution, on="중분류", how="outer")
    .fillna(0)
)

prediction_distribution_compare = prediction_distribution_compare.sort_values(
    "실제_1순위_비율",
    ascending=False,
).round(4)

display(prediction_distribution_compare)